# Push Fixed-Label Data to Hub

Loads new JSONL files generated with the **taxonomy-based fixed vocabulary**, then pushes them as `train_fixed` / `test_fixed` splits into `alexneakameni/ZSHOT-HARDSET-v2`.

**Split strategy**: random 90/10 split by bundle (not label-based), because with ~2.5k fixed labels the label-isolation logic used in the free-label dataset would immediately saturate.

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
HF_DATASET     = "alexneakameni/ZSHOT-HARDSET-v2"
NEW_FILES_PATH = "../data/wikipedia/New/*/wiki*.jsonl"  # <── adjust as needed
TEST_RATIO     = 0.025   # fraction of bundles held out as test
RANDOM_SEED    = 42
# ──────────────────────────────────────────────────────────────────────────────

In [2]:
import json
import random
import datasets
import pandas as pd
from glob import glob
from pathlib import Path

random.seed(RANDOM_SEED)

In [3]:
# ── Load new JSONL files ──────────────────────────────────────────────────────
files = sorted(glob(NEW_FILES_PATH))
print(f"New files found: {len(files)}")
for f in files:
    print(" ", f)

New files found: 1
  ../data/wikipedia/New/Gemma4E4B/wikipedia_synthetic.jsonl


In [4]:
def load_jsonl(file):
    model = Path(file).parent.name
    rows = []
    with open(file) as fh:
        for line in fh:
            d = json.loads(line)
            rows.append({
                "text":       d["text"],
                "labels":     d.get("labels", []),
                "not_labels": d.get("not_labels", []),
                "model":      model,
            })
    return rows

raw = [row for f in files for row in load_jsonl(f)]
df_new = pd.DataFrame(raw)
print(f"Raw rows loaded: {len(df_new)}")

Raw rows loaded: 343634


In [5]:
# ── Merge duplicate texts ─────────────────────────────────────────────────────
def merge_group(group):
    merged_labels     = set().union(*group["labels"])
    merged_not_labels = set().union(*group["not_labels"])
    merged_not_labels -= merged_labels
    models = sorted(set(group["model"]))
    return pd.Series({
        "labels":     sorted(merged_labels),
        "not_labels": sorted(merged_not_labels),
        "model":      models if len(models) > 1 else models[0],
    })

df_new = (
    df_new.groupby("text", sort=False)
    .apply(merge_group, include_groups=False)
    .reset_index()
)
print(f"Unique texts after merging: {len(df_new)}")

Unique texts after merging: 343630


In [6]:
# ── Random 90/10 bundle split ─────────────────────────────────────────────────
# Bundles are groups of 5 consecutive rows (generation order).
# We split at bundle level so no bundle is split across train/test.
BUNDLE_SIZE = 5
n = len(df_new)
n_bundles = n // BUNDLE_SIZE

bundle_indices = list(range(n_bundles))
random.shuffle(bundle_indices)
n_test_bundles  = max(1, int(n_bundles * TEST_RATIO))
test_bundle_set = set(bundle_indices[:n_test_bundles])

row_split = []
for i in range(n):
    b = i // BUNDLE_SIZE
    row_split.append("test" if b in test_bundle_set else "train")

# Any trailing rows that don't form a full bundle go to train
for i in range(n_bundles * BUNDLE_SIZE, n):
    row_split[i] = "train"

df_new["_split"] = row_split

df_fixed_train = df_new[df_new["_split"] == "train"].drop(columns="_split").reset_index(drop=True)
df_fixed_test  = df_new[df_new["_split"] == "test" ].drop(columns="_split").reset_index(drop=True)

print(f"train_fixed: {len(df_fixed_train):,} rows  ({len(df_fixed_train)/n*100:.1f}%)")
print(f"test_fixed : {len(df_fixed_test):,} rows  ({len(df_fixed_test)/n*100:.1f}%)")

train_fixed: 335,040 rows  (97.5%)
test_fixed : 8,590 rows  (2.5%)


In [7]:
# ── Deduplicate against existing fixed splits (if they already exist on Hub) ──
try:
    existing = datasets.load_dataset(HF_DATASET, split=["train_fixed", "test_fixed"])
    existing_train_texts = set(existing[0]["text"])
    existing_test_texts  = set(existing[1]["text"])
    df_fixed_train = df_fixed_train[~df_fixed_train["text"].isin(existing_train_texts)].reset_index(drop=True)
    df_fixed_test  = df_fixed_test[ ~df_fixed_test["text"].isin(existing_test_texts)  ].reset_index(drop=True)
    print(f"After dedup → train_fixed: {len(df_fixed_train):,}  test_fixed: {len(df_fixed_test):,}")
    base_train = existing[0].to_pandas()
    base_test  = existing[1].to_pandas()
    df_fixed_train = pd.concat([base_train, df_fixed_train], ignore_index=True)
    df_fixed_test  = pd.concat([base_test,  df_fixed_test],  ignore_index=True)
    print(f"Total after merge → train_fixed: {len(df_fixed_train):,}  test_fixed: {len(df_fixed_test):,}")
except Exception as e:
    print(f"No existing fixed splits found ({e}) — pushing fresh.")

No existing fixed splits found (Unknown split "train_fixed". Should be one of ['train', 'test'].) — pushing fresh.


In [8]:
# ── Label vocabulary stats ────────────────────────────────────────────────────
train_vocab = set(l for ls in df_fixed_train["labels"] for l in ls) | \
              set(l for ls in df_fixed_train["not_labels"] for l in ls)
test_vocab  = set(l for ls in df_fixed_test["labels"]  for l in ls) | \
              set(l for ls in df_fixed_test["not_labels"]  for l in ls)

print(f"train_fixed label vocab : {len(train_vocab):,}")
print(f"test_fixed  label vocab : {len(test_vocab):,}")
print(f"Labels in test not in train: {len(test_vocab - train_vocab)}")
print(f"Avg pos labels / text   : {df_fixed_train['labels'].apply(len).mean():.2f}")
print(f"Avg neg labels / text   : {df_fixed_train['not_labels'].apply(len).mean():.2f}")

train_fixed label vocab : 24,432
test_fixed  label vocab : 3,780
Labels in test not in train: 256
Avg pos labels / text   : 3.68
Avg neg labels / text   : 12.88


In [9]:
# ── Build DatasetDict and push ────────────────────────────────────────────────
dataset = datasets.DatasetDict({
    "train_fixed": datasets.Dataset.from_pandas(df_fixed_train),
    "test_fixed":  datasets.Dataset.from_pandas(df_fixed_test),
})
dataset

DatasetDict({
    train_fixed: Dataset({
        features: ['text', 'labels', 'not_labels', 'model'],
        num_rows: 335040
    })
    test_fixed: Dataset({
        features: ['text', 'labels', 'not_labels', 'model'],
        num_rows: 8590
    })
})

In [10]:
base_data = datasets.load_dataset(HF_DATASET)

In [11]:
base_data

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'not_labels', 'model'],
        num_rows: 1654452
    })
    test: Dataset({
        features: ['text', 'labels', 'not_labels', 'model'],
        num_rows: 9045
    })
})

In [13]:
full_data = datasets.DatasetDict({
    **base_data,
    **dataset
})

full_data

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'not_labels', 'model'],
        num_rows: 1654452
    })
    test: Dataset({
        features: ['text', 'labels', 'not_labels', 'model'],
        num_rows: 9045
    })
    train_fixed: Dataset({
        features: ['text', 'labels', 'not_labels', 'model'],
        num_rows: 335040
    })
    test_fixed: Dataset({
        features: ['text', 'labels', 'not_labels', 'model'],
        num_rows: 8590
    })
})

In [14]:
full_data.push_to_hub(
    HF_DATASET,
    commit_description=(
        f"Add fixed-vocab splits: {len(df_fixed_train):,} train_fixed rows, "
        f"{len(df_fixed_test):,} test_fixed rows (random {int((1-TEST_RATIO)*100)}/{int(TEST_RATIO*100)} bundle split)."
    ),
)

Uploading the dataset shards:   0%|          | 0/3 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2/commit/c5f8fa81825021939df79ccf61c43485d6c2e856', commit_message='Upload dataset', commit_description='Add fixed-vocab splits: 335,040 train_fixed rows, 8,590 test_fixed rows (random 97/2 bundle split).', oid='c5f8fa81825021939df79ccf61c43485d6c2e856', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alexneakameni/ZSHOT-HARDSET-v2'), pr_revision=None, pr_num=None)